In [1]:
!pip install sentence-transformers faiss-cpu

# AI-Powered Intelligent Hiring Tool Assistant

## Objective
This notebook implements the workflow of the AI-Powered Intelligent Hiring Tool.

It evaluates candidate resumes against a Data Scientist job description using Deep Learning-based semantic similarity and skill matching techniques. It also includes a Retrieval-Augmented Generation (RAG)-based intelligent assistant to provide personalized feedback and answer candidate queries.

### Key Features
- Resume-job semantic matching using Sentence Transformers
- Candidate match score generation
- Skill gap analysis
- Personalized recommendations
- RAG-based AI chatbot
- Interactive Gradio User Interface

In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv('/content/cleaned_resumes.csv')
print(df.shape)
df.head()

(3500, 15)


,ResumeID,Category,Name,Email,Phone,Location,Summary,Skills,Experience,Education,Text,Source,clean_text,resume_length,category_encoded
0,REAL_0001,Java Developer,Chad Griffin,contact@email.com,94105 555 4321000 10 ...,"City, State",jessica claire montgomery street san francisco...,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,ResumeAtlas,jessica claire montgomery street san francisco...,1482,17
1,REAL_0002,Java Developer,Melinda Thomas,contact@email.com,17994568777 2017 2018 20152016 3 ...,"City, State",jared arthur maica java developer 17994568777 ...,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,ResumeAtlas,jared arthur maica java developer linkedincomi...,1666,17
2,REAL_0003,Java Developer,Shannon Mccarthy,contact@email.com,9 555 4321000 94105 8 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,jessica claire resumesampleexamplecom montgome...,5496,17
3,REAL_0004,Java Developer,Christine Kelley,contact@email.com,9 555 4321000 94105 5 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,jessica claire resumesampleexamplecom montgome...,12692,17
4,REAL_0005,Java Developer,Karen Holt,contact@email.com,100 10 4321000 ...,"City, State",jessica claire 100 montgomery st 10th floor xx...,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas,jessica claire montgomery st th floor xxx resu...,4166,17


# Part 1: Deep Learning-Based Resume Matching

In this section, we use a pre-trained Sentence Transformer model to generate semantic embeddings for both the job description and candidate resume.

The embeddings are compared using cosine similarity to calculate a semantic match score.

### Model Used
- SentenceTransformer (all-MiniLM-L6-v2)

### Purpose
To measure how semantically aligned a candidate resume is with the job requirements.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Job Description Setup

We define the target job role and its requirements.

This job description serves as the benchmark against which candidate resumes are evaluated.

### Target Role
Data Scientist

### Core Skills Required
- Python
- SQL
- Machine Learning
- Deep Learning
- TensorFlow
- Pandas
- Statistics

In [5]:
job_description = """
We are hiring a Data Scientist with strong expertise in Python, SQL, Machine Learning, Deep Learning, TensorFlow, Pandas, and Statistics.
The ideal candidate should have experience in data preprocessing, predictive modeling, feature engineering, model evaluation, and deploying machine learning solutions.
Experience with analytics, visualization, and business problem-solving is preferred.
"""

# Part 2: Candidate Evaluation and Skill Gap Analysis

This section evaluates a candidate resume using:
1. Semantic similarity score
2. Skill matching score

The system identifies:
- Match score
- Matched skills
- Missing skills
- Overall verdict

In [ ]:
jd_embedding = model.encode([job_description])

resume_embeddings = model.encode(
    df['clean_text'].tolist(),
    show_progress_bar=True
)

In [7]:
similarities = cosine_similarity(jd_embedding, resume_embeddings).flatten()

semantic_scores_dl = (similarities / similarities.max()) * 100

df['dl_match_score'] = semantic_scores_dl

# Testing the deeplearning model

In [8]:
top_candidates = df.sort_values(
    by='dl_match_score',
    ascending=False
)

top_candidates[['ResumeID', 'Category', 'Skills', 'dl_match_score']].head(10)

,ResumeID,Category,Skills,dl_match_score
2701,SYNTH_2702,Machine Learning Engineer,"Docker, Airflow, MLflow, Git, PyTorch, Keras, ...",100.000000
2681,SYNTH_2682,Machine Learning Engineer,"REST API, Keras, Git, Agile, AWS, PyTorch, MLf...",96.257912
2728,SYNTH_2729,Machine Learning Engineer,"GCP, TensorFlow, PyTorch, Python, Keras, Git, ...",96.017387
423,REAL_0424,Data Science,"Python, SQL, Git, Linux",95.791519
2730,SYNTH_2731,Machine Learning Engineer,"MLflow, Python, Kubernetes, AWS, Airflow, Git,...",95.449364
2704,SYNTH_2705,Machine Learning Engineer,"Docker, PyTorch, MLflow, TensorFlow, Python, K...",94.832130
2683,SYNTH_2684,Machine Learning Engineer,"Microservices, AWS, GCP, TensorFlow, PyTorch, ...",94.635109
2702,SYNTH_2703,Machine Learning Engineer,"PyTorch, Python, Kubernetes, TensorFlow, Airfl...",94.385727
2733,SYNTH_2734,Machine Learning Engineer,"Keras, TensorFlow, MLflow, AWS, Python, REST A...",92.947754
2673,SYNTH_2674,Machine Learning Engineer,"Git, Microservices, TensorFlow, PyTorch, Linux...",91.341743


In [9]:
skill_map = {
    "machine learning": ["machine learning", "ml", "mlflow", "scikit-learn"],
    "deep learning": ["deep learning", "tensorflow", "pytorch", "keras"],
    "python": ["python"],
    "sql": ["sql"],
    "pandas": ["pandas"]
}

skills_required = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pandas"
]

In [10]:
def check_skill(candidate_skills, required_skill):
    candidate_skills = str(candidate_skills).lower()
    aliases = skill_map.get(required_skill, [required_skill])

    for alias in aliases:
        if alias.lower() in candidate_skills:
            return True
    return False

In [11]:
def create_candidate_document(candidate):
    candidate_skills = candidate['Skills']

    matched = []
    missing = []

    for skill in skills_required:
        if check_skill(candidate_skills, skill):
            matched.append(skill)
        else:
            missing.append(skill)

    doc = f"""
Candidate ID: {candidate['ResumeID']}
Category: {candidate['Category']}
Match Score: {candidate['dl_match_score']:.2f}

Matched Skills: {', '.join(matched)}
Missing Skills: {', '.join(missing)}

Recommendation:
Improve missing skills to increase job suitability.
"""
    return doc

In [12]:
top_candidates = df.sort_values(
    by='dl_match_score',
    ascending=False
).head(100)

documents = top_candidates.apply(create_candidate_document, axis=1).tolist()

In [13]:
doc_embeddings = model.encode(documents)

In [14]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype('float32'))

print("Total documents:", index.ntotal)

Total documents: 100


In [15]:
def retrieve(query, k=3):
    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype('float32'),
        k
    )

    results = []
    for idx in indices[0]:
        results.append(documents[idx])

    return results

In [16]:
def candidate_chatbot(query, candidate):
    query = query.lower()

    candidate_skills = candidate['Skills']

    matched = []
    missing = []

    for skill in skills_required:
        if check_skill(candidate_skills, skill):
            matched.append(skill)
        else:
            missing.append(skill)

    score = candidate['dl_match_score']

    if "score" in query:
        return f"""
Your current match score is {score:.2f}%.

This score is calculated using semantic similarity and skill alignment with the job description.
"""

    elif "missing" in query or "weakness" in query:
        return f"""
Based on your profile evaluation, the key missing skills are:

• {', '.join(missing)}

Improving these skills can significantly strengthen your profile.
"""

    elif "improve" in query:
        return f"""
To improve your profile for this role:

1. Strengthen skills in: {', '.join(missing)}
2. Build 2–3 practical projects
3. Add certifications in relevant domains
4. Update your resume with measurable achievements
"""

    elif "why" in query:
        return f"""
Your score depends on:
• Resume-job semantic similarity
• Skill alignment

Matched Skills:
{', '.join(matched)}

Missing Skills:
{', '.join(missing)}
"""

    else:
        return "Please ask about score, missing skills, or improvement areas."

In [17]:
best_candidate = df.sort_values(
    by='dl_match_score',
    ascending=False
).iloc[0]

print(candidate_chatbot("Why is my score low?", best_candidate))
print(candidate_chatbot("What skills are missing?", best_candidate))
print(candidate_chatbot("How can I improve?", best_candidate))


Your current match score is 100.00%.

This score is calculated using semantic similarity and skill alignment with the job description.


Based on your profile evaluation, the key missing skills are:

• pandas

Improving these skills can significantly strengthen your profile.


To improve your profile for this role:

1. Strengthen skills in: pandas
2. Build 2–3 practical projects
3. Add certifications in relevant domains
4. Update your resume with measurable achievements



In [18]:
candidate_resume = input("Paste candidate resume text here:\n")

Paste candidate resume text here:
Amit Verma  Email: [amit.verma@gmail.com](mailto:amit.verma@gmail.com) Location: Noida, India  Professional Summary: Software developer with 2 years of experience in backend development and API design.  Skills: Java, Spring Boot, MySQL, REST API, Git, Linux  Experience: Backend Developer | XYZ Tech | 2022 – Present  * Developed scalable REST APIs. * Worked on backend services and database optimization.  Education: B.Tech in Information Technology


# Skill Matching Logic

Skill matching compares candidate skills with required job skills.

### Purpose
To identify:
- Skills already present in the candidate profile
- Missing skills that need improvement

This enables explainable AI-based candidate evaluation.

In [19]:
resume_embedding = model.encode([candidate_resume])

score = cosine_similarity(jd_embedding, resume_embedding)[0][0]

match_score = ((score + 1) / 2) * 100

print("Candidate Match Score:", round(match_score, 2))

Candidate Match Score: 69.18


In [20]:
matched_skills = []

for skill in skills_required:
    if check_skill(candidate_resume, skill):
        matched_skills.append(skill)

missing_skills = [
    skill for skill in skills_required
    if skill not in matched_skills
]

# Candidate Evaluation Report

This section generates the final evaluation report.

### Output Includes
- Candidate Match Score
- Verdict Classification
- Matched Skills
- Missing Skills

### Verdict Categories
- Strong Match (80%+)
- Moderate Match (65–79%)
- Weak Match (<65%)

In [21]:
score = match_score

if score >= 80:
    verdict = "Strong Match"
elif score >= 65:
    verdict = "Moderate Match"
else:
    verdict = "Weak Match"

print("===== Candidate Evaluation =====")
print("Match Score:", round(match_score, 2))
print("Verdict:", verdict)
print("Matched Skills:", matched_skills)
print("Missing Skills:", missing_skills)

===== Candidate Evaluation =====
Match Score: 69.18
Verdict: Moderate Match
Matched Skills: ['sql']
Missing Skills: ['python', 'machine learning', 'deep learning', 'pandas']


In [22]:
candidate_profile = {
    "match_score": match_score,
    "skills": matched_skills
}

# Part 3: Retrieval-Augmented Generation (RAG) Candidate Assistant

This section implements an intelligent assistant that answers candidate queries based on evaluation results.

The assistant provides:
- Personalized responses
- Explainable recommendations
- Improvement suggestions

### Example Queries
- What is my score?
- What skills are missing?
- How can I improve?
- Why is my score low?

In [23]:
def user_chatbot(query, candidate_profile, missing_skills):
    query = query.lower()

    score = candidate_profile['match_score']

    # Verdict classification
    if score >= 80:
        verdict = "Strong Match"
    elif score >= 65:
        verdict = "Moderate Match"
    else:
        verdict = "Weak Match"

    # SCORE QUERY
    if "score" in query:
        return f"""
Your match score is {score:.2f}%.

Verdict: {verdict}

This score is calculated using:
• Semantic similarity between your resume and job description
• Skill alignment with required technical skills
"""

    # MISSING SKILLS QUERY
    elif "missing" in query or "skills" in query:
        if len(missing_skills) == 0:
            return """
Excellent! You currently have all required core skills for this role.
"""
        else:
            return f"""
Your missing skills are:
{', '.join(missing_skills)}

Improving these skills can significantly strengthen your profile.
"""

    # IMPROVEMENT QUERY
    elif "improve" in query:
        if len(missing_skills) == 0:
            return """
Your profile is already strongly aligned with this role.

To further improve:
1. Build advanced real-world projects
2. Add industry-recognized certifications
3. Gain more practical experience
4. Strengthen resume achievements with measurable results
"""
        else:
            return f"""
To improve your profile:

1. Learn {', '.join(missing_skills)}
2. Build practical projects in these domains
3. Add certifications
4. Update resume with measurable achievements
"""

    # WHY QUERY
    elif "why" in query:
        if len(missing_skills) == 0:
            return f"""
Your profile is already strong with a match score of {score:.2f}%.

Your score is mainly determined by:
• Resume-job semantic similarity
• Technical skill match
• Overall relevance to job requirements
"""
        else:
            return f"""
Your score is impacted by missing skills in:

{', '.join(missing_skills)}

These missing areas reduce alignment with job requirements.
"""

    # VERDICT QUERY
    elif "verdict" in query or "result" in query:
        return f"""
Final Evaluation: {verdict}
Match Score: {score:.2f}%
"""

    else:
        return """
I can help you with candidate evaluation.

You can ask:
• What is my score?
• What skills are missing?
• How can I improve?
• Why is my score low?
• What is my verdict?
"""

# Part 4: Interactive User Interface

To make the system user-friendly, we build an interactive Gradio interface.

The UI allows candidates to:
- Enter their details
- Upload/paste resume text
- View evaluation results
- Ask questions to the AI assistant

In [24]:
!pip install gradio

In [25]:
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_candidate(name, resume_text, query):

    # Generate Resume Embedding
    resume_embedding = model.encode([resume_text])

    # Semantic Score
    score = cosine_similarity(jd_embedding, resume_embedding)[0][0]
    match_score = ((score + 1) / 2) * 100

    # Skill Matching
    matched_skills = []

    for skill in skills_required:
        if check_skill(resume_text, skill):
            matched_skills.append(skill)

    missing_skills = [
        skill for skill in skills_required
        if skill not in matched_skills
    ]

    # Candidate Profile
    candidate_profile = {
        "match_score": match_score,
        "skills": matched_skills
    }

    # Verdict
    if match_score >= 80:
        verdict = "Strong Match"
    elif match_score >= 65:
        verdict = "Moderate Match"
    else:
        verdict = "Weak Match"

    # Chatbot Response
    bot_response = user_chatbot(
        query,
        candidate_profile,
        missing_skills
    )

    # Final Result
    result = f"""
Candidate Name: {name}
Role: Data Scientist

Match Score: {match_score:.2f}%
Verdict: {verdict}

Matched Skills: {', '.join(matched_skills) if matched_skills else 'None'}
Missing Skills: {', '.join(missing_skills) if missing_skills else 'None'}
"""

    return result, bot_response

In [26]:
demo = gr.Interface(
    fn=evaluate_candidate,
    inputs=[
        gr.Textbox(label="Candidate Name"),
        gr.Textbox(lines=15, label="Paste Resume Text"),
        gr.Textbox(label="Ask AI Assistant")
    ],
    outputs=[
        gr.Textbox(label="Candidate Evaluation"),
        gr.Textbox(label="AI Assistant Response")
    ],
    title="AI-Powered Intelligent Hiring Tool",
    description="""
Data Scientist Role Evaluation

Job Description:
We are hiring a Data Scientist with strong expertise in Python, SQL, Machine Learning, Deep Learning, TensorFlow, Pandas, and Statistics.

The ideal candidate should have experience in:
• Data preprocessing
• Predictive modeling
• Feature engineering
• Model evaluation
• Deploying machine learning solutions

Preferred:
• Analytics
• Visualization
• Business problem-solving
"""
)

In [27]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3039ffcbba15a2221.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Part 5: Deployment

The final application is deployed using Hugging Face Spaces. It is live at: https://huggingface.co/spaces/MDBITW/ai-hiring-tool

### Deployment Platform
- Gradio
- Hugging Face Spaces

### Benefits
- Live demo access
- Interactive candidate evaluation
- Real-time AI assistant responses

# Final Project Summary

This notebook completes the candidate-side implementation of the AI-Powered Intelligent Hiring Tool.

### Components Implemented
- Deep Learning-based semantic scoring
- Skill gap analysis
- Candidate evaluation system
- RAG-based chatbot
- Interactive UI deployment

### Technologies Used
- Python
- Sentence Transformers
- Scikit-learn
- Gradio
- Hugging Face

This system helps candidates understand their suitability for a role and receive actionable recommendations for improvement.